# Description de séries temporelles par LLM : valeurs brutes, features, contraste et agent
**Ghada Dakhlaoui** – projet personnel (Kaggle, GPU T4)

Question : un petit LLM décrit-il mieux un signal quand on lui donne (A) les valeurs brutes,
(B) des features statistiques, (C) des features + comparaison à des signaux de référence (contraste),
ou (D) quand il pilote lui-même des outils d'analyse (agent) ?
Évaluation sur des signaux **synthétiques dont la vérité terrain est connue** (tendance, périodicité, anomalie).
Les prompts sont réglés sur un jeu DEV ; le jeu TEST n'est utilisé qu'une seule fois à la fin.

## Étape 0 – Installation et configuration

In [ ]:
import os, sys, json, re, time, math, random, subprocess
USE_MOCK = os.environ.get("USE_MOCK") == "1"      # test local sans LLM
if not USE_MOCK:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.44", "accelerate"], check=False)
import numpy as np, pandas as pd
from scipy.stats import linregress
import matplotlib
if USE_MOCK: matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 42
L = 500                      # longueur des signaux
N_DEV, N_TEST, N_REF = 30, 200, 100
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"       # option : "Qwen/Qwen2.5-3B-Instruct"
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./out"
os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
print("OUT =", OUT, "| MOCK =", USE_MOCK)


## Étape 1 – Signaux synthétiques avec vérité terrain
Chaque signal combine : une tendance (monte / descend / plate), une composante périodique (oui/non, nombre de cycles),
une anomalie (un pic isolé, oui/non) et un bruit gaussien.

In [ ]:
def make_signal(rng, L=L):
    t = np.arange(L) / L
    trend = str(rng.choice(["up", "down", "flat"]))
    periodic = bool(rng.integers(0, 2))
    anomaly = bool(rng.integers(0, 2))
    noise = rng.uniform(0.1, 0.3)
    x = rng.normal(0, noise, L)
    if trend != "flat":
        x += (1 if trend == "up" else -1) * rng.uniform(1.0, 2.5) * t
    cycles = None
    if periodic:
        cycles = int(rng.integers(4, 13))
        x += rng.uniform(0.6, 1.2) * np.sin(2 * np.pi * cycles * t + rng.uniform(0, 2 * np.pi))
    pos = None
    if anomaly:
        pos = int(rng.integers(50, L - 50))
        w = int(rng.integers(1, 4))
        x[pos:pos + w] += rng.choice([-1, 1]) * rng.uniform(3, 5)
    return x, dict(trend=trend, periodic=periodic, anomaly=anomaly, cycles=cycles, anomaly_pos=pos)

def make_set(n, seed):
    rng = np.random.default_rng(seed)
    return [make_signal(rng) for _ in range(n)]

def make_reference(n, seed):          # signaux « normaux » : plats, non périodiques, sans anomalie
    rng = np.random.default_rng(seed)
    return [rng.normal(0, rng.uniform(0.1, 0.3), L) for _ in range(n)]

DEV = make_set(N_DEV, SEED + 1)
TEST = make_set(N_TEST, SEED + 2)
REF = make_reference(N_REF, SEED + 3)
print(len(DEV), len(TEST), len(REF))


## Étape 2 – Features statistiques (NumPy / SciPy)

In [ ]:
def extract_features(x):
    n = len(x); t = np.arange(n)
    lr = linregress(t, x)
    detr = x - (lr.intercept + lr.slope * t)
    med = np.median(detr); mad = 1.4826 * np.median(np.abs(detr - med)) + 1e-9
    clipped = np.clip(detr, med - 6 * mad, med + 6 * mad)          # limite l'effet des pics sur le spectre
    P = np.abs(np.fft.rfft(clipped - clipped.mean())) ** 2
    lo, hi = 2, n // 10
    band = P[lo:hi + 1]
    k = lo + int(np.argmax(band)); ratio = float(band.max() / P[1:].sum())
    resid = detr
    if ratio > 0.25:                                               # retire la composante périodique dominante
        A = np.c_[np.sin(2 * np.pi * k * t / n), np.cos(2 * np.pi * k * t / n)]
        coef, *_ = np.linalg.lstsq(A, detr, rcond=None)
        resid = detr - A @ coef
    med2 = np.median(resid); mad2 = 1.4826 * np.median(np.abs(resid - med2)) + 1e-9
    z = np.abs(resid - med2) / mad2
    return dict(length=n, mean=float(x.mean()), std=float(x.std()), min=float(x.min()), max=float(x.max()),
                slope_total=float(lr.slope * n), r2_trend=float(lr.rvalue ** 2),
                dominant_cycles=k, spectral_peak_ratio=ratio,
                noise_std=float(mad2), resid_max_z=float(z.max()), resid_max_pos_frac=float(z.argmax() / n),
                n_outliers=int((z > 6).sum()))

def rule_baseline(f):                  # référence sans LLM : seuils simples
    tr = "up" if f["slope_total"] > 0.5 else "down" if f["slope_total"] < -0.5 else "flat"
    return dict(trend=tr, periodic=f["spectral_peak_ratio"] > 0.25, anomaly=f["resid_max_z"] > 6)

REF_F = pd.DataFrame([extract_features(x) for x in REF])
REF_STATS = {k: (float(REF_F[k].mean()), float(REF_F[k].std())) for k in ["slope_total", "spectral_peak_ratio", "resid_max_z"]}
print(REF_STATS)

def acc_of(preds, truth, key):
    return float(np.mean([p is not None and p.get(key) == g[key] for p, (x, g) in zip(preds, truth)]))
FEATS_DEV = [extract_features(x) for x, g in DEV]
rb = [rule_baseline(f) for f in FEATS_DEV]
print("Baseline à seuils (DEV) :", {k: round(acc_of(rb, DEV, k), 3) for k in ["trend", "periodic", "anomaly"]})


## Étape 3 – Les quatre méthodes (prompts) et les outils de l'agent

In [ ]:
FORMAT = ('Answer with ONE JSON object only: {"trend": "up" or "down" or "flat", "periodic": true or false, '
          '"anomaly": true or false, "description": "one short sentence"}. '
          '"anomaly" means an isolated spike or outlier that does not belong to the trend or the periodic pattern.')
SYS_PLAIN = "You are an expert in time series analysis. " + FORMAT

def prompt_raw(x):         # A : toutes les valeurs brutes
    vals = ", ".join(f"{v:.1f}" for v in x)
    return [{"role": "system", "content": SYS_PLAIN},
            {"role": "user", "content": f"Time series of {len(x)} points:\n[{vals}]\nDescribe it."}]

def features_text(f):
    return (f"- length: {f['length']}\n- mean: {f['mean']:.2f}, std: {f['std']:.2f}, min: {f['min']:.2f}, max: {f['max']:.2f}\n"
            f"- slope_total (total change of the linear fit over the whole window, signal units): {f['slope_total']:+.2f}\n"
            f"- r2_trend (share of variance explained by the linear trend): {f['r2_trend']:.2f}\n"
            f"- noise_std (robust std of the residual after removing trend and periodic part): {f['noise_std']:.2f}\n"
            f"- dominant_cycles (cycles per window of the strongest oscillation): {f['dominant_cycles']}\n"
            f"- spectral_peak_ratio (share of oscillation energy in that frequency, 0 to 1): {f['spectral_peak_ratio']:.2f}\n"
            f"- resid_max_z (largest residual deviation, in robust standard deviations): {f['resid_max_z']:.1f}\n"
            f"- n_outliers (residual points beyond 6 robust standard deviations): {f['n_outliers']}")

REF_RANGE = {k: (float(REF_F[k].min()), float(REF_F[k].max())) for k in ["slope_total", "spectral_peak_ratio", "resid_max_z"]}

def _cmp(k, v):
    lo, hi = REF_RANGE[k]
    pos = "ABOVE" if v > hi else "BELOW" if v < lo else "WITHIN"
    return f"{k} = {v:.2f}; normal reference range [{lo:.2f}, {hi:.2f}] -> {pos} the normal range"

def contrast_text(f):
    return "Comparison with normal reference signals (flat, non-periodic, no anomaly):\n" + "\n".join("- " + _cmp(k, f[k]) for k in REF_RANGE)

def prompt_features(f):    # B
    return [{"role": "system", "content": SYS_PLAIN},
            {"role": "user", "content": "Measured features of a time series:\n" + features_text(f) + "\nDescribe it."}]

def prompt_contrast(f):    # C
    return [{"role": "system", "content": SYS_PLAIN},
            {"role": "user", "content": "Measured features of a time series:\n" + features_text(f) + "\n\n" + contrast_text(f) + "\nDescribe it."}]

# --- D : agent avec outils -----------------------------------------------------
TOOLS = ["trend", "periodicity", "anomaly"]
def run_tool(name, f):
    m = {"trend": ["slope_total", "r2_trend", "noise_std"], "periodicity": ["dominant_cycles", "spectral_peak_ratio"],
         "anomaly": ["resid_max_z", "n_outliers", "resid_max_pos_frac"]}[name]
    txt = f"[{name}] " + "; ".join(f"{k} = {f[k]:.2f}" if isinstance(f[k], float) else f"{k} = {f[k]}" for k in m)
    for k in m:
        if k in REF_RANGE: txt += " | " + _cmp(k, f[k])
    return txt

SYS_AGENT = ("You are a time series analysis agent. You never see the values. You have three tools: trend, periodicity, anomaly. "
             "Each returns measurements and a comparison with normal reference signals (flat, non-periodic, no anomaly). "
             'Reply with ONE JSON object per turn: either {"action": "call_tool", "tool": "trend"|"periodicity"|"anomaly"} '
             'or, when you have enough evidence, {"action": "final", "trend": "up"|"down"|"flat", "periodic": true|false, '
             '"anomaly": true|false, "description": "one short sentence"}. Call each tool you need before the final answer.')

def parse_json(txt):
    m = re.search(r"\{.*?\}", txt, re.S)
    if not m: return None
    try: return json.loads(m.group(0))
    except Exception: return None

def norm_answer(d):
    if not isinstance(d, dict): return None
    def b(v):
        if isinstance(v, bool): return v
        if isinstance(v, str) and v.strip().lower() in ("true", "yes"): return True
        if isinstance(v, str) and v.strip().lower() in ("false", "no"): return False
        return None
    tr = str(d.get("trend", "")).strip().lower()
    out = dict(trend=tr if tr in ("up", "down", "flat") else None, periodic=b(d.get("periodic")), anomaly=b(d.get("anomaly")),
               description=str(d.get("description", ""))[:300])
    return out if all(out[k] is not None for k in ("trend", "periodic", "anomaly")) else None


## Étape 4 – Chargement du LLM (ou faux LLM pour tester le code)

In [ ]:
class MockLLM:
    """Faux LLM pour tester le pipeline sans GPU : lit les mesures dans le prompt."""
    def __call__(self, messages, max_new_tokens=220):
        sysm = messages[0]["content"]; allu = "\n".join(m["content"] for m in messages[1:])
        def num(name):
            m = re.search(name + r"[^\n=:]*[=:] ?(-?\+?[0-9.]+)", allu); return float(m.group(1)) if m else None
        if "call_tool" in sysm:
            done = [t for t in TOOLS if f"[{t}]" in allu]
            if len(done) < 3: return json.dumps({"action": "call_tool", "tool": [t for t in TOOLS if t not in done][0]})
        s, r, z = num("slope_total"), num("spectral_peak_ratio"), num("resid_max_z")
        if s is None: return json.dumps({"trend": "flat", "periodic": False, "anomaly": False, "description": "no info"})
        d = dict(trend="up" if s > 0.5 else "down" if s < -0.5 else "flat", periodic=bool(r and r > 0.25), anomaly=bool(z and z > 6),
                 description="mock")
        if "call_tool" in sysm: d["action"] = "final"
        return json.dumps(d)

if USE_MOCK:
    llm = MockLLM()
else:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    assert torch.cuda.is_available(), "Active le GPU (Settings > Accelerator > GPU T4)"
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="auto")
    model.eval()
    def llm(messages, max_new_tokens=220):
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc = tok(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
        return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
print("LLM prêt :", "MOCK" if USE_MOCK else MODEL_NAME)


## Étape 5 – Exécution des méthodes et de l'agent

In [ ]:
def run_agent(f, max_turns=5):
    summ = f"Summary: length={f['length']}, mean={f['mean']:.2f}, std={f['std']:.2f}, min={f['min']:.2f}, max={f['max']:.2f}. Choose a tool."
    msgs = [{"role": "system", "content": SYS_AGENT}, {"role": "user", "content": summ}]
    called, fallback = [], False
    for _ in range(max_turns):
        raw = llm(msgs); d = parse_json(raw)
        msgs.append({"role": "assistant", "content": raw})
        if d and d.get("action") == "call_tool" and d.get("tool") in TOOLS and d["tool"] not in called:
            called.append(d["tool"])
            msgs.append({"role": "user", "content": "Tool result: " + run_tool(d["tool"], f) + "\nNext action?"})
            continue
        ans = norm_answer(d) if d and d.get("action") == "final" else None
        if ans: return ans, called, fallback
        break
    fallback = True                                   # repli : on exécute les outils manquants puis réponse forcée
    for t in TOOLS:
        if t not in called:
            called.append(t); msgs.append({"role": "user", "content": "Tool result: " + run_tool(t, f)})
    msgs.append({"role": "user", "content": 'Now give the final JSON: {"action": "final", "trend": ..., "periodic": ..., "anomaly": ..., "description": ...}'})
    return norm_answer(parse_json(llm(msgs))), called, fallback

def evaluate(data, name, show=0):
    rows, preds, n_tools, fb = [], [], [], 0
    t0 = time.time()
    for i, (x, g) in enumerate(data):
        f = extract_features(x)
        if name == "A_raw": p = norm_answer(parse_json(llm(prompt_raw(x))))
        elif name == "B_features": p = norm_answer(parse_json(llm(prompt_features(f))))
        elif name == "C_contrast": p = norm_answer(parse_json(llm(prompt_contrast(f))))
        elif name == "D_agent":
            p, called, fallback = run_agent(f); n_tools.append(len(called)); fb += int(fallback)
        elif name == "R_rules": p = rule_baseline(f)
        preds.append(p)
        if i < show: print(name, "| vérité:", {k: g[k] for k in ("trend", "periodic", "anomaly")}, "| prédit:", p)
    dt = (time.time() - t0) / len(data)
    n = len(data); res = dict(method=name, n=n, parse_fail=sum(p is None for p in preds) / n, sec_per_signal=dt)
    for k in ("trend", "periodic", "anomaly"): res[k] = acc_of(preds, data, k)
    res["exact_match"] = float(np.mean([p is not None and all(p[k] == g[k] for k in ("trend", "periodic", "anomaly")) for p, (x, g) in zip(preds, data)]))
    if n_tools: res["avg_tool_calls"] = float(np.mean(n_tools)); res["fallback_rate"] = fb / n
    return res, preds

METHODS = ["R_rules", "A_raw", "B_features", "C_contrast", "D_agent"]


### 5a – Essai sur le jeu DEV (30 signaux) : sert à vérifier que les prompts fonctionnent
On regarde 3 exemples par méthode et le taux d'échec de lecture JSON. **On ne touche plus aux prompts après le TEST.**

In [ ]:
dev_rows = []
for m in METHODS:
    r, _ = evaluate(DEV, m, show=3 if m != "R_rules" else 0); dev_rows.append(r)
dev_df = pd.DataFrame(dev_rows); print(dev_df.round(3).to_string(index=False))
dev_df.to_csv(f"{OUT}/results_dev.csv", index=False)


## Étape 6 – Évaluation finale sur le jeu TEST (200 signaux, une seule exécution)

In [ ]:
test_rows, test_preds = [], {}
for m in METHODS:
    r, p = evaluate(TEST, m); test_rows.append(r); test_preds[m] = p
test_df = pd.DataFrame(test_rows)
for k in ["trend", "periodic", "anomaly", "exact_match"]:
    test_df[k + "_ci95"] = 1.96 * np.sqrt(test_df[k] * (1 - test_df[k]) / test_df["n"])
test_df.to_csv(f"{OUT}/results_test.csv", index=False)
print(test_df.round(3).to_string(index=False))
with open(f"{OUT}/predictions_test.json", "w") as fh:
    json.dump({m: [p for p in ps] for m, ps in test_preds.items()}, fh, default=str)


## Étape 7 – Figures

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
w = 0.16; keys = ["trend", "periodic", "anomaly", "exact_match"]
for i, m in enumerate(METHODS):
    row = test_df[test_df.method == m].iloc[0]
    ax.bar(np.arange(len(keys)) + i * w, [row[k] for k in keys], w, yerr=[row[k + "_ci95"] for k in keys], label=m, capsize=2)
ax.set_xticks(np.arange(len(keys)) + 2 * w); ax.set_xticklabels(["Tendance", "Périodicité", "Anomalie", "Les 3 corrects"])
ax.set_ylim(0, 1.05); ax.set_ylabel("Précision (TEST, n=200)"); ax.legend(fontsize=8, ncol=3); ax.set_title(f"Description de signaux – {MODEL_NAME if not USE_MOCK else 'MOCK'}")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_comparaison.png", dpi=150); plt.show()

fig, axs = plt.subplots(2, 3, figsize=(12, 5))
for a, (x, g) in zip(axs.ravel(), TEST[:6]):
    a.plot(x, lw=0.8); a.set_title(f"{g['trend']} | périodique={g['periodic']} | anomalie={g['anomaly']}", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_exemples.png", dpi=150); plt.show()


## Étape 8 – Analyse des erreurs
Quelles erreurs fait chaque méthode, et sur quel attribut ?

In [ ]:
for m in METHODS[1:]:
    errs = {"trend": 0, "periodic": 0, "anomaly": 0, "illisible": 0}
    for p, (x, g) in zip(test_preds[m], TEST):
        if p is None: errs["illisible"] += 1; continue
        for k in ("trend", "periodic", "anomaly"): errs[k] += int(p[k] != g[k])
    print(m, errs)
print("\nRésumé complet dans", OUT)
